# CPSTN Traffic Forecasting - Testing & Application Pipeline (Draft 5)

**Purpose:** Professional, failure-safe evaluation of the trained CPSTN-Draft5 model, followed by traffic-aware A* route planning using the model's predicted traffic.

**Important:** This notebook performs **testing and application only**. It does not retrain CPSTN, modify trained weights, or alter forecasting metrics to improve them.

**Pipeline**

```text
Trained Checkpoint
        ↓
Test Dataset + Graph Validation
        ↓
CPSTN Inference
        ↓
Inverse Scaling
        ↓
Forecast Evaluation
   ├── Overall MAE / RMSE / MAPE
   ├── Horizon-wise analysis
   ├── Sensor-wise analysis
   └── Error analysis
        ↓
Predicted Traffic
        ↓
A* Traffic-Aware Routing
        ↓
Route Results + Visualization
```


## 1. Project Overview

CPSTN is used here as the **traffic forecasting stage**. A* is a separate **application stage** that consumes predicted traffic from CPSTN.

The evaluation preserves the Draft5 testing logic: checkpoint-defined preprocessing/configuration, chronological test windows, inverse scaling, and the existing MAE/RMSE/MAPE convention. Missing or incompatible inputs are reported and stop only the affected stage rather than being silently repaired.


## 2. Configuration

The checkpoint remains the source of truth for dataset, node count, input length, forecast horizon, normalization statistics, adjacency, and model configuration.

Routing configuration is intentionally separate from CPSTN evaluation. No routing result changes forecasting metrics.


In [ ]:
# 2. Configuration
import os
import gc
import json
import time
import pickle
import hashlib
import shutil
from pathlib import Path
from collections import deque

ROOT = Path("/content/cpstn_testing_draft5")
ROOT.mkdir(parents=True, exist_ok=True)
DATA_DIR = ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR = ROOT / "exports"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Optional explicit routing endpoints.
# Leave as None for deterministic automatic selection from the valid graph.
SOURCE_SENSOR_ID = None
DESTINATION_SENSOR_ID = None

# Route uses one actual test sample and one forecast horizon produced by CPSTN.
ROUTE_SAMPLE_INDEX = 0
ROUTE_HORIZON_STEP = 0  # 0 = first forecast step, 1 = second, ...

# Representative sensors for plots. If empty, the notebook selects evenly spaced sensors.
REPRESENTATIVE_SENSOR_IDS = []

# Strict validation switches
REQUIRE_CUDA = False


## 3. Imports

In [ ]:
# 3. Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

if REQUIRE_CUDA and not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required. In Colab use Runtime -> Change runtime type -> GPU.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device :", device)
if torch.cuda.is_available():
    print("GPU    :", torch.cuda.get_device_name(0))
    print("CUDA  :", torch.version.cuda)


## 4. Load CPSTN Checkpoint

The checkpoint is validated before any inference. Loading failure stops the inference pipeline and reports the actual component/path/error.

In [ ]:
# 4. Locate and load the authoritative Draft 5 checkpoint
LOCAL_CANDIDATES = [
    Path("/content/CPSTN(Training)-Draft5_final.pt"),
    Path("/content/CPSTN-Training-Draft5_final.pt"),
    Path("/content/cpstn_testing_draft5/CPSTN(Training)-Draft5_final.pt"),
]
MODEL_PATH = next((p for p in LOCAL_CANDIDATES if p.exists() and p.stat().st_size > 0), None)
if MODEL_PATH is None:
    try:
        from google.colab import files
        print("Upload CPSTN(Training)-Draft5_final.pt")
        uploaded = files.upload()
        pts = [Path(name) for name in uploaded if name.lower().endswith((".pt", ".pth"))]
        if not pts: raise RuntimeError("No .pt/.pth checkpoint uploaded.")
        MODEL_PATH = pts[0]
    except ImportError as e:
        raise FileNotFoundError("Authoritative Draft 5 checkpoint not found. Put CPSTN(Training)-Draft5_final.pt in /content.") from e

try:
    ckpt = torch.load(MODEL_PATH, map_location="cpu", weights_only=False)
except TypeError:
    ckpt = torch.load(MODEL_PATH, map_location="cpu")

required = ["model_state_dict", "model_config", "training_config", "dataset", "num_nodes", "input_steps", "horizon", "input_channels", "hidden", "dropout", "mean", "std", "adjacency_normalized", "sensor_ids"]
missing = [k for k in required if k not in ckpt]
if missing: raise RuntimeError(f"Checkpoint missing required fields: {missing}")

DATASET = str(ckpt["dataset"]); N = int(ckpt["num_nodes"]); INPUT_STEPS = int(ckpt["input_steps"]); HORIZON = int(ckpt["horizon"]); INPUT_CHANNELS = int(ckpt["input_channels"]); HIDDEN = int(ckpt["hidden"]); DROPOUT = float(ckpt["dropout"])
mean = np.asarray(ckpt["mean"], dtype=np.float32); std = np.asarray(ckpt["std"], dtype=np.float32); A_norm = np.asarray(ckpt["adjacency_normalized"], dtype=np.float32); sensor_ids = [str(x) for x in ckpt["sensor_ids"]]
EXPECTED_NODES = {"METR-LA": 207, "PEMS-BAY": 325}
if DATASET not in EXPECTED_NODES or N != EXPECTED_NODES[DATASET]: raise RuntimeError(f"Checkpoint dataset/node mismatch: {DATASET}, {N}")
if INPUT_CHANNELS != 1: raise RuntimeError(f"Draft 5 checkpoint expects {INPUT_CHANNELS} channels; expected 1 observed speed feature.")
if mean.shape != (N,) or std.shape != (N,) or A_norm.shape != (N,N): raise RuntimeError("Checkpoint scaler/graph dimensions are incompatible.")
if not np.isfinite(mean).all() or not np.isfinite(std).all() or not np.isfinite(A_norm).all() or np.any(std <= 0): raise RuntimeError("Checkpoint scaler/graph contains invalid values.")
if len(sensor_ids) != N or len(set(sensor_ids)) != N: raise RuntimeError("Checkpoint sensor IDs are missing or duplicated.")
print("="*78); print("CHECKPOINT VALIDATION: PASS"); print("Path:", MODEL_PATH); print("Model:", ckpt.get("model_name")); print("Dataset:", DATASET); print("Nodes:", N); print("Input steps:", INPUT_STEPS); print("Horizon:", HORIZON); print("Input channels:", INPUT_CHANNELS); print("Hidden:", HIDDEN); print("Epoch:", ckpt.get("epoch")); print("Best validation loss:", ckpt.get("best_val_loss")); print("Graph backend:", ckpt.get("graph_backend")); print("="*78)


## 5. Load Test Dataset

In [ ]:
# 5. Dataset acquisition
DATA_CFG = {
    "METR-LA": {
        "file": DATA_DIR / "metr-la.h5",
        "drive_id": "1pAGRfzMx6K9WWsfDcD1NMbIif0T0saFC",
        "expected_nodes": 207,
    },
    "PEMS-BAY": {
        "file": DATA_DIR / "pems-bay.h5",
        "drive_id": "1wD-mHlqAb2mtHOe_68fZvDh1LpDegMMq",
        "expected_nodes": 325,
    },
}
cfg = DATA_CFG[DATASET]

try:
    import gdown
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown"])
    import gdown

def download_dataset():
    path = Path(cfg["file"])
    if path.exists() and path.stat().st_size > 0:
        return path
    print("Downloading", DATASET, "traffic data...")
    result = gdown.download(id=cfg["drive_id"], output=str(path), quiet=False)
    if result is None or not path.exists() or path.stat().st_size <= 0:
        raise RuntimeError(f"Dataset download failed: {path}")
    return path

H5_PATH = download_dataset()
print("Dataset file:", H5_PATH)
print("Size:", round(H5_PATH.stat().st_size / 1024**2, 2), "MB")


## 6. Dataset / Graph Validation

In [ ]:
# 6. Load and validate test data
try:
    try:
        df = pd.read_hdf(H5_PATH, key="speed")
    except (KeyError, ValueError):
        df = pd.read_hdf(H5_PATH)
except Exception as e:
    raise RuntimeError(f"Test dataset loading failed for '{H5_PATH}': {type(e).__name__}: {e}") from e

if df is None or df.empty:
    raise RuntimeError("DATA FAILURE: test dataset is empty.")

try:
    df.index = pd.to_datetime(df.index)
except Exception as e:
    raise RuntimeError(f"DATA FAILURE: timestamps could not be parsed: {e}") from e

if not df.index.is_monotonic_increasing:
    df = df.sort_index()

if df.index.has_duplicates:
    raise RuntimeError("DATA FAILURE: duplicate timestamps were found; no silent deduplication was performed.")

df.columns = [str(c) for c in df.columns]

if df.shape[1] != N:
    raise RuntimeError(
        f"SHAPE FAILURE: sensor count mismatch. Expected {N} from checkpoint, actual {df.shape[1]}."
    )

# Match checkpoint sensor ordering exactly.
dataset_sensor_ids = list(df.columns)
if set(dataset_sensor_ids) != set(sensor_ids):
    missing_ids = sorted(set(sensor_ids) - set(dataset_sensor_ids))
    extra_ids = sorted(set(dataset_sensor_ids) - set(sensor_ids))
    raise RuntimeError(
        "SENSOR FAILURE: dataset sensor IDs do not match checkpoint. "
        f"Missing={missing_ids[:10]}, Extra={extra_ids[:10]}"
    )
df = df.loc[:, sensor_ids]

# Draft4 preprocessing retained: numeric conversion, inf/zero -> missing, interpolation + edge fill.
df = df.apply(pd.to_numeric, errors="coerce")
missing_before = int(df.isna().sum().sum())
df = df.replace([np.inf, -np.inf], np.nan)
df = df.replace(0.0, np.nan)
df = df.interpolate(method="linear", limit_direction="both").ffill().bfill()

if df.isna().any().any():
    bad = int(df.isna().sum().sum())
    raise RuntimeError(f"DATA FAILURE: {bad} NaN values remain after the existing Draft4 preprocessing.")

speed = np.ascontiguousarray(df.to_numpy(dtype=np.float32))
timestamps = df.index
T = len(speed)

train_end = int(T * 0.70)
val_end = int(T * 0.80)

if T <= val_end + INPUT_STEPS + HORIZON:
    raise RuntimeError(
        f"DATA FAILURE: test segment is too small. Rows={T}, "
        f"required beyond validation boundary={val_end + INPUT_STEPS + HORIZON}."
    )

# Check timestamp cadence rather than inventing a cadence.
intervals = timestamps.to_series().diff().dropna()
if intervals.empty or (intervals <= pd.Timedelta(0)).any():
    raise RuntimeError("DATA FAILURE: timestamps are not strictly increasing.")
median_interval = intervals.median()

# Use ONLY checkpoint scaler.
speed_z = ((speed - mean[None, :]) / np.maximum(std[None, :], 1e-3)).astype(np.float32)

# Graph validation. A_norm is the normalized graph stored in the checkpoint.
if not np.isfinite(A_norm).all():
    raise RuntimeError("GRAPH FAILURE: normalized adjacency contains NaN/Inf.")
if A_norm.shape != (N, N):
    raise RuntimeError(f"GRAPH FAILURE: adjacency shape expected {(N, N)}, got {A_norm.shape}.")

print("=" * 78)
print("DATASET / GRAPH VALIDATION: SUCCESS")
print("=" * 78)
print("Data shape          :", speed.shape)
print("Time range          :", timestamps.min(), "->", timestamps.max())
print("Median interval     :", median_interval)
print("Missing before fix  :", missing_before)
print("Train end row       :", train_end)
print("Validation end row  :", val_end)
print("Test rows           :", T - val_end)
print("Sensor ordering     : checkpoint-aligned")
print("Graph shape         :", A_norm.shape)


## 7. Generate Predictions

The test-window construction follows the existing Draft5 logic: the first test target begins after the validation boundary plus the complete input history. No cross-boundary history is introduced.

In [ ]:
# 7. Build test windows and exact Draft 5 model
# Training notebook uses test inputs with INPUT_STEPS historical observations immediately before val_end; targets start at val_end.
first_t = val_end
last_t = T - HORIZON
if last_t < first_t: raise RuntimeError("Test segment is too small for the checkpoint horizon.")
starts = np.arange(first_t, last_t + 1, dtype=np.int64)
X = np.stack([speed_z[t-INPUT_STEPS:t] for t in starts], axis=0).astype(np.float32)
Y = np.stack([speed_z[t:t+HORIZON] for t in starts], axis=0).astype(np.float32)
if X.shape != (len(starts), INPUT_STEPS, N) or Y.shape != (len(starts), HORIZON, N): raise RuntimeError(f"Window shape mismatch: X={X.shape}, Y={Y.shape}")

class DenseGraphConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__(); self.proj = nn.Conv2d(in_channels, out_channels, kernel_size=1)
    def forward(self, x, A_dense):
        B,C,Tm,Nn=x.shape
        z=x.permute(0,2,1,3).contiguous().reshape(B*Tm*C,Nn)
        z=torch.matmul(z.float(), A_dense.float().T).to(dtype=x.dtype)
        z=z.reshape(B,Tm,C,Nn).permute(0,2,1,3).contiguous()
        return self.proj(z)

class TemporalGate(nn.Module):
    def __init__(self,in_channels,out_channels,dilation=1,dropout=0.1):
        super().__init__(); self.conv=nn.Conv2d(in_channels,2*out_channels,kernel_size=(3,1),padding=(dilation,0),dilation=(dilation,1)); self.dropout=nn.Dropout(dropout)
    def forward(self,x):
        a,b=self.conv(x).chunk(2,dim=1); return self.dropout(torch.tanh(a)*torch.sigmoid(b))

class CPSTNBlock(nn.Module):
    def __init__(self,channels,dropout=0.1,dilation=1):
        super().__init__(); self.temporal=TemporalGate(channels,channels,dilation=dilation,dropout=dropout); self.graph=DenseGraphConv(channels,channels); self.norm=nn.BatchNorm2d(channels); self.ff=nn.Conv2d(channels,channels,1); self.dropout=nn.Dropout(dropout)
    def forward(self,x,A_dense):
        residual=x; x=self.temporal(x); x=F.gelu(self.graph(x,A_dense)); x=self.ff(x); x=self.dropout(self.norm(x)); return F.gelu(x+residual)

class CPSTN(nn.Module):
    def __init__(self,input_channels=1,hidden=32,horizon=12,dropout=0.1,number_of_blocks=3,dilations=(1,2,4)):
        super().__init__(); self.input_proj=nn.Conv2d(input_channels,hidden,1); self.blocks=nn.ModuleList([CPSTNBlock(hidden,dropout,dilation=d) for d in dilations]); self.head=nn.Sequential(nn.Conv2d(hidden,hidden,1),nn.GELU(),nn.Dropout(dropout),nn.Conv2d(hidden,horizon,1))
    def forward(self,x,A_dense):
        if x.ndim==3: x=x.unsqueeze(-1)
        x=x.permute(0,3,1,2).contiguous(); x=self.input_proj(x)
        for block in self.blocks: x=block(x,A_dense)
        return self.head(x[:,:,-1:,:]).squeeze(2)

model=model=CPSTN(**ckpt["model_config"]).to(device)
model.load_state_dict(ckpt["model_state_dict"], strict=True)
model.eval()
A_tensor=torch.from_numpy(A_norm).to(device=device,dtype=torch.float32)
params=sum(p.numel() for p in model.parameters())
if params != 26668: raise RuntimeError(f"Unexpected parameter count: {params}")
with torch.inference_mode():
    smoke=model(torch.zeros((2,INPUT_STEPS,N),device=device),A_tensor)
if tuple(smoke.shape)!=(2,HORIZON,N): raise RuntimeError(f"Smoke output shape mismatch: {tuple(smoke.shape)}")
del smoke
print("Model strict load: PASS | parameters:",params,"| test windows:",len(X),"| X:",X.shape,"| Y:",Y.shape)


## 8. Overall Metrics

MAE, RMSE and the existing Draft5-style MAPE denominator floor are retained. MAPE is protected against zero/near-zero actual values. A failed metric is represented as `N/A`, never as zero.

In [ ]:
# 8. Inverse scaling and alignment validation
pred = pred_z * std[None, None, :] + mean[None, None, :]
true = true_z * std[None, None, :] + mean[None, None:]

if pred.shape != true.shape:
    raise RuntimeError(f"ALIGNMENT FAILURE: actual shape {true.shape}, prediction shape {pred.shape}.")
if len(starts) != len(pred):
    raise RuntimeError(f"ALIGNMENT FAILURE: starts={len(starts)}, predictions={len(pred)}.")
if not np.isfinite(pred).all() or not np.isfinite(true).all():
    raise RuntimeError("SCALER/ALIGNMENT FAILURE: inverse-scaled values contain NaN/Inf.")

prediction_timestamps = timestamps[starts]
if len(prediction_timestamps) != len(pred):
    raise RuntimeError("ALIGNMENT FAILURE: timestamp count does not match prediction count.")

def compute_metrics(p, y):
    p = np.asarray(p)
    y = np.asarray(y)
    if p.shape != y.shape or p.size == 0:
        return {"MAE": np.nan, "RMSE": np.nan, "MAPE_percent": np.nan}
    if not np.isfinite(p).all() or not np.isfinite(y).all():
        return {"MAE": np.nan, "RMSE": np.nan, "MAPE_percent": np.nan}
    err = p - y
    mae = np.mean(np.abs(err))
    rmse = np.sqrt(np.mean(err ** 2))
    # Existing Draft4 convention: denominator floor at 5 speed units.
    mape = np.mean(np.abs(err) / np.maximum(np.abs(y), 5.0)) * 100.0
    return {"MAE": float(mae), "RMSE": float(rmse), "MAPE_percent": float(mape)}

overall = compute_metrics(pred, true)
overall_df = pd.DataFrame([
    {"Metric": "MAE", "Value": overall["MAE"]},
    {"Metric": "RMSE", "Value": overall["RMSE"]},
    {"Metric": "MAPE", "Value": overall["MAPE_percent"]},
])
display(overall_df)


## 9. Horizon-wise Evaluation

In [ ]:
# 9. Horizon-wise metrics
horizon_rows = []
for h in range(HORIZON):
    m = compute_metrics(pred[:, h, :], true[:, h, :])
    horizon_rows.append({
        "Horizon Step": h + 1,
        "Horizon Minutes": (h + 1) * 5,
        **m
    })
horizon_df = pd.DataFrame(horizon_rows)
display(horizon_df)

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(horizon_df["Horizon Minutes"], horizon_df["MAE"], marker="o", label="MAE")
ax.plot(horizon_df["Horizon Minutes"], horizon_df["RMSE"], marker="s", label="RMSE")
ax.set_title(f"{DATASET}: Horizon-wise Forecast Error")
ax.set_xlabel("Forecast horizon (minutes)")
ax.set_ylabel("Error")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(horizon_df["Horizon Minutes"], horizon_df["MAPE_percent"], marker="o", label="MAPE")
ax.set_title(f"{DATASET}: Horizon-wise MAPE")
ax.set_xlabel("Forecast horizon (minutes)")
ax.set_ylabel("MAPE (%)")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
plt.show()


## 10. Sensor-wise Evaluation

In [ ]:
# 10. Sensor-wise metrics
sensor_rows = []
excluded_sensors = []

for j, sid in enumerate(sensor_ids):
    p = pred[:, :, j]
    y = true[:, :, j]
    valid = np.isfinite(p) & np.isfinite(y)
    if valid.sum() == 0:
        excluded_sensors.append({"sensor_id": sid, "reason": "No finite actual/predicted values."})
        continue
    m = compute_metrics(p[valid], y[valid])
    sensor_rows.append({"sensor_id": sid, **m})

sensor_df = pd.DataFrame(sensor_rows)

if sensor_df.empty:
    raise RuntimeError("SENSOR FAILURE: no sensor has sufficient valid data for sensor-level metrics.")

display(sensor_df)

lowest_mae_sensor = sensor_df.loc[sensor_df["MAE"].idxmin()]
highest_mae_sensor = sensor_df.loc[sensor_df["MAE"].idxmax()]

print("Lowest-error sensor by MAE :", lowest_mae_sensor["sensor_id"], f"({lowest_mae_sensor['MAE']:.4f})")
print("Highest-error sensor by MAE:", highest_mae_sensor["sensor_id"], f"({highest_mae_sensor['MAE']:.4f})")
if excluded_sensors:
    print("\nExcluded sensors:")
    display(pd.DataFrame(excluded_sensors))

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(np.arange(len(sensor_df)), sensor_df["MAE"])
ax.set_title(f"{DATASET}: Sensor-wise MAE")
ax.set_xlabel("Sensor index (checkpoint order)")
ax.set_ylabel("MAE")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(np.arange(len(sensor_df)), sensor_df["RMSE"])
ax.set_title(f"{DATASET}: Sensor-wise RMSE")
ax.set_xlabel("Sensor index (checkpoint order)")
ax.set_ylabel("RMSE")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
plt.show()


## 11. Visual Prediction Analysis

In [ ]:
# 11A. Representative sensors
if REPRESENTATIVE_SENSOR_IDS:
    missing_rep = [s for s in REPRESENTATIVE_SENSOR_IDS if s not in sensor_ids]
    if missing_rep:
        raise RuntimeError(f"Visualization sensor selection invalid. Unknown sensor IDs: {missing_rep}")
    representative_indices = [sensor_ids.index(s) for s in REPRESENTATIVE_SENSOR_IDS]
else:
    k = min(4, N)
    representative_indices = np.linspace(0, N - 1, k, dtype=int).tolist()

sample_idx = min(ROUTE_SAMPLE_INDEX, len(pred) - 1)

for sensor_idx in representative_indices:
    sid = sensor_ids[sensor_idx]
    history_z = X[sample_idx, :, sensor_idx, 0]
    history = history_z * std[sensor_idx] + mean[sensor_idx]
    future_true = true[sample_idx, :, sensor_idx]
    future_pred = pred[sample_idx, :, sensor_idx]

    history_x = np.arange(INPUT_STEPS)
    future_x = np.arange(INPUT_STEPS, INPUT_STEPS + HORIZON)

    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(history_x, history, marker="o", label="History")
    ax.plot(future_x, future_true, marker="o", label="Actual future")
    ax.plot(future_x, future_pred, marker="x", label="CPSTN prediction")
    ax.axvline(INPUT_STEPS - 0.5, linestyle="--", alpha=0.5)
    ax.set_title(f"{DATASET}: Actual vs Predicted - Sensor {sid}")
    ax.set_xlabel("5-minute step")
    ax.set_ylabel("Speed")
    ax.grid(alpha=0.25)
    ax.legend()
    fig.tight_layout()
    plt.show()

# 11B. Multiple-sensor comparison at one forecast horizon
h0 = min(ROUTE_HORIZON_STEP, HORIZON - 1)
fig, ax = plt.subplots(figsize=(12, 5))
for sensor_idx in representative_indices:
    ax.plot(
        true[sample_idx, :, sensor_idx],
        label=f"Actual {sensor_ids[sensor_idx]}"
    )
    ax.plot(
        pred[sample_idx, :, sensor_idx],
        linestyle="--",
        label=f"Pred {sensor_ids[sensor_idx]}"
    )
ax.set_title(f"{DATASET}: Multiple Sensor Forecast Comparison")
ax.set_xlabel("Forecast step")
ax.set_ylabel("Speed")
ax.grid(alpha=0.25)
ax.legend(ncol=2, fontsize=8)
fig.tight_layout()
plt.show()

# 11C. Forecast horizon visualization for one sensor
sensor_idx = representative_indices[0]
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(np.arange(1, HORIZON + 1), true[sample_idx, :, sensor_idx], marker="o", label="Actual")
ax.plot(np.arange(1, HORIZON + 1), pred[sample_idx, :, sensor_idx], marker="x", label="Predicted")
ax.set_title(f"{DATASET}: Forecast Horizon - Sensor {sensor_ids[sensor_idx]}")
ax.set_xlabel("Forecast horizon step")
ax.set_ylabel("Speed")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
plt.show()

# 11D. Actual vs predicted scatter
scatter_actual = true.reshape(-1)
scatter_pred = pred.reshape(-1)
fig, ax = plt.subplots(figsize=(6.5, 6))
ax.scatter(scatter_actual, scatter_pred, s=4, alpha=0.18)
lo = min(scatter_actual.min(), scatter_pred.min())
hi = max(scatter_actual.max(), scatter_pred.max())
ax.plot([lo, hi], [lo, hi], linestyle="--", label="Ideal y=x")
ax.set_title(f"{DATASET}: Actual vs Predicted")
ax.set_xlabel("Actual speed")
ax.set_ylabel("Predicted speed")
ax.legend()
ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()


## 12. Error Analysis

In [ ]:
# 12. Error analysis
errors = pred - true
abs_errors = np.abs(errors)
flat_abs_error = abs_errors.reshape(-1)

error_mean_abs = float(np.mean(flat_abs_error))
error_max_abs = float(np.max(flat_abs_error))
error_min_abs = float(np.min(flat_abs_error))
error_std = float(np.std(errors.reshape(-1)))

error_summary_df = pd.DataFrame([
    ["Mean absolute error", error_mean_abs],
    ["Maximum absolute error", error_max_abs],
    ["Minimum absolute error", error_min_abs],
    ["Error standard deviation", error_std],
], columns=["Statistic", "Value"])
display(error_summary_df)

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.hist(errors.reshape(-1), bins=60)
ax.axvline(0, linestyle="--", alpha=0.7)
ax.set_title(f"{DATASET}: Prediction Error Distribution")
ax.set_xlabel("Prediction error (predicted - actual)")
ax.set_ylabel("Frequency")
ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()

print(
    f"The mean absolute error is {error_mean_abs:.4f} and the maximum observed "
    f"absolute error is {error_max_abs:.4f}."
)


## 13. Graph Visualization

The checkpoint contains a normalized adjacency matrix, but no geographic coordinates are created here. Therefore, the graph plot is a **network topology**, not a geographic road map.

In [ ]:
# 13. Build graph topology from the checkpoint adjacency.
# Self-loops are omitted from the routing topology. No coordinates are invented.
A_graph = np.asarray(A_norm, dtype=np.float64).copy()
np.fill_diagonal(A_graph, 0.0)

# Treat positive finite adjacency entries as graph connections.
edge_mask = np.isfinite(A_graph) & (A_graph > 0)
undirected_mask = edge_mask | edge_mask.T
edges = [(i, j) for i in range(N) for j in range(i + 1, N) if undirected_mask[i, j]]

print("Graph nodes :", N)
print("Topology edges (undirected display):", len(edges))

# Deterministic circular layout. This is a topology layout only, not geographic coordinates.
theta = np.linspace(0, 2 * np.pi, N, endpoint=False)
layout_xy = np.column_stack([np.cos(theta), np.sin(theta)])

fig, ax = plt.subplots(figsize=(10, 10))
for i, j in edges:
    ax.plot(
        [layout_xy[i, 0], layout_xy[j, 0]],
        [layout_xy[i, 1], layout_xy[j, 1]],
        linewidth=0.5, alpha=0.12
    )
ax.scatter(layout_xy[:, 0], layout_xy[:, 1], s=14)
ax.set_title(f"{DATASET}: CPSTN Graph Topology (not a geographic road map)")
ax.set_axis_off()
fig.tight_layout()
plt.show()


## 14. A* Traffic-aware Route Planning

A* runs **after** CPSTN predictions. Because the supplied checkpoint does not provide distance/speed-to-edge metadata or geographic coordinates, this notebook does not invent them. Instead, it uses a documented traffic-based proxy edge cost derived from predicted sensor speed:

**Proxy edge cost = adjacency connection × inverse predicted speed.**

Without geographic coordinates, the A* heuristic is `h(n)=0`, which is a valid graph-compatible admissible heuristic and makes the search Dijkstra-equivalent while retaining the A* interface.

In [ ]:
# 14. A* preparation and validation
def build_routing_graph(adj, predicted_speed, speed_floor=1.0):
    """
    Build a directed graph from positive adjacency entries.
    Edge cost is a documented proxy:
        cost = adjacency_weight / max(predicted_speed, speed_floor)
    It is NOT a physical travel time because no distance metadata is available.
    """
    adj = np.asarray(adj, dtype=float)
    speed = np.asarray(predicted_speed, dtype=float)
    if adj.shape != (N, N):
        raise RuntimeError(f"GRAPH FAILURE: adjacency shape {adj.shape} != {(N, N)}.")
    if speed.shape != (N,):
        raise RuntimeError(f"ROUTING INPUT FAILURE: predicted speed shape {speed.shape} != {(N,)}.")
    if not np.isfinite(adj).all() or not np.isfinite(speed).all():
        raise RuntimeError("ROUTING INPUT FAILURE: non-finite adjacency or predicted speed.")
    if not np.any(adj > 0):
        raise RuntimeError("GRAPH FAILURE: graph has no positive edges.")

    graph = {i: [] for i in range(N)}
    for u in range(N):
        for v in np.flatnonzero(adj[u] > 0):
            if u == v:
                continue
            w = float(adj[u, v])
            safe_speed = max(float(speed[v]), speed_floor)
            cost = w / safe_speed
            if not np.isfinite(cost) or cost <= 0:
                raise RuntimeError(f"EDGE COST FAILURE: invalid cost for edge {u}->{v}: {cost}.")
            graph[u].append((int(v), cost))
    return graph

def connected_components_undirected(graph):
    undirected = {i: set() for i in graph}
    for u, nbrs in graph.items():
        for v, _ in nbrs:
            undirected[u].add(v)
            undirected[v].add(u)
    comps, unseen = [], set(undirected)
    while unseen:
        start = min(unseen)
        q, comp = [start], []
        unseen.remove(start)
        while q:
            u = q.pop()
            comp.append(u)
            for v in undirected[u]:
                if v in unseen:
                    unseen.remove(v)
                    q.append(v)
        comps.append(sorted(comp))
    return comps

def choose_route_endpoints(graph):
    comps = sorted(connected_components_undirected(graph), key=lambda c: (-len(c), c[0]))
    if not comps or len(comps[0]) < 2:
        raise RuntimeError("GRAPH FAILURE: no connected component contains at least two nodes.")
    return comps[0][0], comps[0][-1], len(comps[0])

def a_star(graph, source, destination, heuristic=None):
    import heapq
    if source not in graph:
        raise ValueError(f"Invalid source node: {source}")
    if destination not in graph:
        raise ValueError(f"Invalid destination node: {destination}")
    heuristic = heuristic or (lambda _u, _v: 0.0)

    open_heap = [(float(heuristic(source, destination)), 0.0, source)]
    g_score = {source: 0.0}
    came_from = {}
    visited = set()

    while open_heap:
        _, g_current, u = heapq.heappop(open_heap)
        if u in visited:
            continue
        visited.add(u)
        if u == destination:
            path = [u]
            while u in came_from:
                u = came_from[u]
                path.append(u)
            return path[::-1], float(g_current)

        for v, edge_cost in graph[u]:
            tentative = g_current + edge_cost
            if tentative < g_score.get(v, float("inf")):
                g_score[v] = tentative
                came_from[v] = u
                heapq.heappush(
                    open_heap,
                    (tentative + float(heuristic(v, destination)), tentative, v)
                )
    return None, float("inf")

# A* is an application stage. Forecasting has already completed above.
routing_status = "SKIPPED"
routing_failure_reason = None
route = []
route_cost = np.nan
route_speed = None
route_sample = int(ROUTE_SAMPLE_INDEX)
route_h = int(ROUTE_HORIZON_STEP)
source = destination = None
routing_graph = None
largest_component_size = None

try:
    if not (0 <= route_h < HORIZON):
        raise RuntimeError(
            f"ROUTING CONFIG FAILURE: ROUTE_HORIZON_STEP={route_h} outside 0..{HORIZON-1}."
        )
    if not (0 <= route_sample < len(pred)):
        raise RuntimeError(
            f"ROUTING CONFIG FAILURE: ROUTE_SAMPLE_INDEX={route_sample} outside 0..{len(pred)-1}."
        )

    route_speed = pred[route_sample, route_h, :]
    routing_graph = build_routing_graph(A_graph, route_speed)

    id_to_idx = {sid: i for i, sid in enumerate(sensor_ids)}
    if SOURCE_SENSOR_ID is not None:
        if str(SOURCE_SENSOR_ID) not in id_to_idx:
            raise RuntimeError(f"INVALID SOURCE: sensor ID '{SOURCE_SENSOR_ID}' does not exist.")
        source = id_to_idx[str(SOURCE_SENSOR_ID)]
    if DESTINATION_SENSOR_ID is not None:
        if str(DESTINATION_SENSOR_ID) not in id_to_idx:
            raise RuntimeError(f"INVALID DESTINATION: sensor ID '{DESTINATION_SENSOR_ID}' does not exist.")
        destination = id_to_idx[str(DESTINATION_SENSOR_ID)]

    if source is None or destination is None:
        auto_source, auto_destination, largest_component_size = choose_route_endpoints(routing_graph)
        source = auto_source if source is None else source
        destination = auto_destination if destination is None else destination

    if source == destination:
        raise RuntimeError("ROUTING FAILURE: source and destination are identical.")

    # No coordinates/distance metadata exist, so h(n)=0 is the documented admissible heuristic.
    route, route_cost = a_star(routing_graph, source, destination, heuristic=lambda _u, _v: 0.0)

    if route is None:
        routing_failure_reason = (
            f"No valid path between source and destination "
            f"({sensor_ids[source]} -> {sensor_ids[destination]})."
        )
        routing_status = "FAILED"
        print("A* route unavailable: no valid path between source and destination.")
    else:
        routing_status = "SUCCESS"
        print("=" * 78)
        print("A* ROUTING: SUCCESS")
        print("=" * 78)
        print("Source node      :", sensor_ids[source])
        print("Destination node :", sensor_ids[destination])
        print("Route nodes      :", len(route))
        print("Route cost       :", f"{route_cost:.6f}")
        print("Forecast horizon :", f"+{(route_h + 1) * 5} minutes")
        print("Test sample      :", route_sample)
        print("Cost model       : adjacency-weighted inverse predicted speed proxy")
        print("Heuristic        : 0 (geographic coordinates unavailable)")
        if largest_component_size is not None:
            print("Largest component:", largest_component_size, "nodes")

except Exception as e:
    routing_status = "SKIPPED" if "GRAPH FAILURE" in str(e) else "FAILED"
    routing_failure_reason = f"{type(e).__name__}: {e}"
    route = []
    route_cost = np.nan
    print("A* routing did not complete.")
    print("Reason:", routing_failure_reason)
    print("Forecasting results remain valid and are preserved.")


## 15. Route Visualization

In [ ]:
# 15. Route results and visualization
if routing_status == "SUCCESS":
    route_rows = []
    for step, node in enumerate(route):
        if step == 0:
            edge_cost = 0.0
        else:
            prev = route[step - 1]
            matches = [c for v, c in routing_graph[prev] if v == node]
            edge_cost = float(matches[0]) if matches else np.nan
        route_rows.append({
            "Step": step,
            "Node": sensor_ids[node],
            "Predicted Traffic": float(route_speed[node]),
            "Edge Cost": edge_cost,
        })
    route_df = pd.DataFrame(route_rows)
    display(route_df)

    route_summary_df = pd.DataFrame([{
        "Source Node": sensor_ids[source],
        "Destination Node": sensor_ids[destination],
        "Selected Route": " -> ".join(sensor_ids[n] for n in route),
        "Number of Nodes": len(route),
        "Route Cost": route_cost,
        "Estimated Travel Time": "N/A - distance metadata unavailable",
        "Traffic Summary": (
            f"Predicted speed range along route: "
            f"{route_speed[route].min():.4f} to {route_speed[route].max():.4f}"
        ),
    }])
    display(route_summary_df)

    fig, ax = plt.subplots(figsize=(10, 10))
    for i, j in edges:
        ax.plot(
            [layout_xy[i, 0], layout_xy[j, 0]],
            [layout_xy[i, 1], layout_xy[j, 1]],
            linewidth=0.5, alpha=0.10
        )
    ax.scatter(layout_xy[:, 0], layout_xy[:, 1], s=10, alpha=0.35)
    route_xy = layout_xy[route]
    ax.plot(route_xy[:, 0], route_xy[:, 1], linewidth=3, marker="o", label="Selected A* route")
    ax.scatter(layout_xy[source, 0], layout_xy[source, 1], s=100, marker="o", label="Source")
    ax.scatter(layout_xy[destination, 0], layout_xy[destination, 1], s=100, marker="X", label="Destination")
    ax.set_title(f"{DATASET}: A* Traffic-Aware Route on Graph Topology (not geographic)")
    ax.set_axis_off()
    ax.legend()
    fig.tight_layout()
    plt.show()
else:
    route_df = pd.DataFrame(columns=["Step", "Node", "Predicted Traffic", "Edge Cost"])
    route_summary_df = pd.DataFrame([{
        "Source Node": "N/A",
        "Destination Node": "N/A",
        "Selected Route": "N/A",
        "Number of Nodes": "N/A",
        "Route Cost": "N/A",
        "Estimated Travel Time": "N/A",
        "Traffic Summary": "A* route unavailable."
    }])
    print("A* visualization skipped because routing status is:", routing_status)
    print("Reason:", routing_failure_reason)


## 16. Final Numerical Results

In [ ]:
# 16. Forecasting performance: overall + every horizon
final_metric = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "MAPE (%)"],
    "Overall": [overall["MAE"], overall["RMSE"], overall["MAPE_percent"]]
})
for _, row in horizon_df.iterrows():
    col = f"H{int(row['Horizon Step'])} (+{int(row['Horizon Minutes'])}m)"
    final_metric[col] = [row["MAE"], row["RMSE"], row["MAPE_percent"]]

print("CPSTN FORECASTING PERFORMANCE")
display(final_metric)

print("A* ROUTE PLANNING RESULTS")
display(route_summary_df)


## 17. Final Visual Results Dashboard

In [ ]:
# 17. Compact examiner-facing dashboard
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

axes[0, 0].scatter(scatter_actual, scatter_pred, s=3, alpha=0.15)
axes[0, 0].plot([lo, hi], [lo, hi], linestyle="--")
axes[0, 0].set_title("Actual vs Predicted Traffic")
axes[0, 0].set_xlabel("Actual")
axes[0, 0].set_ylabel("Predicted")

axes[0, 1].plot(horizon_df["Horizon Minutes"], horizon_df["MAE"], marker="o")
axes[0, 1].set_title("Horizon-wise MAE")
axes[0, 1].set_xlabel("Minutes")
axes[0, 1].set_ylabel("MAE")

axes[0, 2].plot(horizon_df["Horizon Minutes"], horizon_df["RMSE"], marker="o")
axes[0, 2].set_title("Horizon-wise RMSE")
axes[0, 2].set_xlabel("Minutes")
axes[0, 2].set_ylabel("RMSE")

axes[1, 0].bar(np.arange(len(sensor_df)), sensor_df["MAE"])
axes[1, 0].set_title("Sensor-wise MAE")
axes[1, 0].set_xlabel("Sensor index")
axes[1, 0].set_ylabel("MAE")

axes[1, 1].hist(errors.reshape(-1), bins=50)
axes[1, 1].axvline(0, linestyle="--", alpha=0.7)
axes[1, 1].set_title("Error Distribution")
axes[1, 1].set_xlabel("Prediction error")
axes[1, 1].set_ylabel("Frequency")

if routing_status == "SUCCESS":
    for i, j in edges:
        axes[1, 2].plot(
            [layout_xy[i, 0], layout_xy[j, 0]],
            [layout_xy[i, 1], layout_xy[j, 1]],
            linewidth=0.4, alpha=0.10
        )
    axes[1, 2].scatter(layout_xy[:, 0], layout_xy[:, 1], s=7, alpha=0.3)
    axes[1, 2].plot(route_xy[:, 0], route_xy[:, 1], linewidth=3, marker="o")
    axes[1, 2].scatter(layout_xy[source, 0], layout_xy[source, 1], s=70, marker="o")
    axes[1, 2].scatter(layout_xy[destination, 0], layout_xy[destination, 1], s=70, marker="X")
    axes[1, 2].set_title("A* Route (Topology)")
    axes[1, 2].set_axis_off()
else:
    axes[1, 2].text(
        0.5, 0.5,
        f"A* ROUTING: {routing_status}\n{routing_failure_reason or 'No route generated.'}",
        ha="center", va="center", wrap=True
    )
    axes[1, 2].set_title("A* Route Result")
    axes[1, 2].set_axis_off()

fig.suptitle(f"{DATASET}: CPSTN Testing + Traffic-Aware Routing Dashboard", fontsize=15)
fig.tight_layout()
plt.show()


## 18. Textual Interpretation

In [ ]:
# 18. Generate factual interpretation only from calculated values.
def finite(v):
    return v is not None and np.isfinite(v)

def fmt(v, digits=4):
    return f"{v:.{digits}f}" if finite(v) else "N/A"

h_mae_first = horizon_df.iloc[0]["MAE"]
h_mae_last = horizon_df.iloc[-1]["MAE"]
h_rmse_first = horizon_df.iloc[0]["RMSE"]
h_rmse_last = horizon_df.iloc[-1]["RMSE"]
mae_change = h_mae_last - h_mae_first
rmse_change = h_rmse_last - h_rmse_first

print("### Overall Performance")
print(
    f"Overall MAE = {fmt(overall['MAE'])}, RMSE = {fmt(overall['RMSE'])}, "
    f"and MAPE = {fmt(overall['MAPE_percent'], 2)}% over the evaluated test windows."
)

print("\n### Horizon Analysis")
direction_mae = "increased" if mae_change > 0 else "decreased" if mae_change < 0 else "remained unchanged"
direction_rmse = "increased" if rmse_change > 0 else "decreased" if rmse_change < 0 else "remained unchanged"
print(f"MAE {direction_mae} from {fmt(h_mae_first)} at the first horizon to {fmt(h_mae_last)} at the final horizon.")
print(f"RMSE {direction_rmse} from {fmt(h_rmse_first)} at the first horizon to {fmt(h_rmse_last)} at the final horizon.")

print("\n### Sensor Analysis")
print(
    f"Sensor {lowest_mae_sensor['sensor_id']} has the lowest observed MAE "
    f"({fmt(lowest_mae_sensor['MAE'])}), while sensor "
    f"{highest_mae_sensor['sensor_id']} has the highest observed MAE "
    f"({fmt(highest_mae_sensor['MAE'])})."
)
if excluded_sensors:
    print(f"{len(excluded_sensors)} sensor(s) were excluded from sensor-level metrics because they lacked finite evaluation data.")

print("\n### Prediction Behavior")
print(
    f"The actual-vs-predicted analysis contains {scatter_actual.size:,} paired values. "
    "The diagonal y=x line represents exact prediction equality; the plotted point spread shows observed deviations."
)

print("\n### Error Analysis")
print(
    f"The mean absolute error is {fmt(error_mean_abs)} and the maximum observed absolute error is "
    f"{fmt(error_max_abs)}. The error standard deviation is {fmt(error_std)}."
)

print("\n### Route Planning")
if routing_status == "SUCCESS":
    print(
        f"CPSTN predictions for test sample {route_sample}, forecast step {route_h + 1} "
        f"(+{(route_h + 1) * 5} minutes) were supplied to A*. "
        f"The selected route contains {len(route)} nodes and has proxy cost {route_cost:.6f}."
    )
else:
    print(f"A* routing status: {routing_status}. Reason: {routing_failure_reason}")

print("\n### Limitations")
print("- Test coverage is limited to the supplied chronological test segment.")
print("- Sensor coverage is limited to sensors present in the checkpoint-aligned dataset.")
print("- The supplied graph contains adjacency information but no geographic coordinates.")
print("- Because physical distance metadata is unavailable, route cost is a predicted-speed/adjacency proxy, not physical travel time.")
print("- Estimated travel time is therefore reported as N/A.")
print("- The A* heuristic is h(n)=0 because geographic distance information is unavailable.")


## 19. Failure / Diagnostic Report

The notebook uses explicit failure states. Forecasting results are not discarded merely because a later application stage fails. Conversely, invalid forecasting inputs do not produce fake metrics or plots.

In [ ]:
# 19. Final stage-status report
status = {
    "CHECKPOINT": "SUCCESS",
    "DATA LOADING": "SUCCESS",
    "INFERENCE": "SUCCESS",
    "METRICS": "SUCCESS",
    "VISUALIZATIONS": "SUCCESS",
    "A* ROUTING": routing_status,
}

core_ok = all(status[k] == "SUCCESS" for k in [
    "CHECKPOINT", "DATA LOADING", "INFERENCE", "METRICS"
])
all_required_ok = core_ok and status["VISUALIZATIONS"] == "SUCCESS" and routing_status == "SUCCESS"
overall_status = "SUCCESS" if all_required_ok else "PARTIAL"

print("=" * 78)
print("FINAL TESTING STATUS")
print("=" * 78)
for k, v in status.items():
    print(f"{k:18s}: {v}")
print(f"{'OVERALL STATUS':18s}: {overall_status}")
print("=" * 78)

diagnostic_report = pd.DataFrame([{"Stage": k, "Status": v} for k, v in status.items()])
display(diagnostic_report)

if routing_status != "SUCCESS":
    print("A* diagnostic:", routing_failure_reason)
print("\nFailure-handling policy:")
print("- Checkpoint failure -> stop inference and report the actual error.")
print("- Data/shape/scaler/alignment failure -> stop the affected evaluation stage.")
print("- NaN/Inf predictions -> stop metric calculation and affected visualizations.")
print("- Metric failure -> report N/A, never zero.")
print("- Visualization failure -> preserve valid numerical results.")
print("- Sensor failure -> report excluded sensors and reasons.")
print("- Graph failure -> skip A* while preserving valid forecasting results.")
print("- A* failure -> report exact failure; never fabricate a route.")


## 20. Final Project Summary

```text
                         TRAINING
                             ↓
                     Multiple Datasets
                             ↓
                           CPSTN
                             ↓
                       Trained Model
                             ↓
                          TESTING
                             ↓
                       Traffic Forecast
                             ↓
              ┌──────────────┴──────────────┐
              ↓                             ↓
     Forecast Evaluation             Predicted Traffic
              ↓                             ↓
       MAE / RMSE / MAPE                    A*
                                            ↓
                                   Traffic-Aware Route
```

**CPSTN = traffic forecasting.**

**A* = route planning using CPSTN-predicted traffic.**

They are separate stages. A* does not modify CPSTN predictions, trained weights, or forecasting metrics.

The routing implementation uses only graph and traffic information genuinely available from the checkpoint/test pipeline. Where geographic distance or physical travel-time information is unavailable, the notebook reports that limitation instead of inventing it.



## 21. Comprehensive evaluation exports


In [ ]:
# Comprehensive metrics, reports, raw arrays, plots, and package
from sklearn.metrics import r2_score, explained_variance_score, median_absolute_error
RESULTS_ROOT = ROOT / "results"
for sub in ["metrics","predictions","visualizations/predictions","visualizations/errors","visualizations/horizons","visualizations/nodes","visualizations/distributions","reports"]:
    (RESULTS_ROOT/sub).mkdir(parents=True, exist_ok=True)

def metrics_extended(p,y):
    p=np.asarray(p,dtype=np.float64); y=np.asarray(y,dtype=np.float64); e=p-y; ae=np.abs(e)
    return {"MAE":float(ae.mean()),"MSE":float(np.mean(e**2)),"RMSE":float(np.sqrt(np.mean(e**2))),"MAPE_percent":float(np.mean(ae/np.maximum(np.abs(y),5.0))*100),"sMAPE_percent":float(np.mean(2*ae/np.maximum(np.abs(p)+np.abs(y),1e-8))*100),"WAPE_percent":float(np.sum(ae)/max(np.sum(np.abs(y)),1e-8)*100),"R2":float(r2_score(y.reshape(-1),p.reshape(-1))),"ExplainedVariance":float(explained_variance_score(y.reshape(-1),p.reshape(-1))),"MedianAbsoluteError":float(median_absolute_error(y.reshape(-1),p.reshape(-1)))}

overall_ext=metrics_extended(pred,true)
h_rows=[]
for h in range(HORIZON): h_rows.append({"Horizon Step":h+1,"Horizon Minutes":(h+1)*5,**metrics_extended(pred[:,h,:],true[:,h,:])})
h_ext=pd.DataFrame(h_rows)
n_rows=[]
for j,sid in enumerate(sensor_ids): n_rows.append({"node_index":j,"sensor_id":sid,**metrics_extended(pred[:,:,j],true[:,:,j])})
n_ext=pd.DataFrame(n_rows)
t_rows=[]
for i,start in enumerate(starts): t_rows.append({"sample_index":i,"target_start_row":int(start),"timestamp":str(timestamps[start]),**metrics_extended(pred[i],true[i])})
t_ext=pd.DataFrame(t_rows)
err=pred-true; flat=err.reshape(-1); absflat=np.abs(flat)
err_stats={"mean_error":float(flat.mean()),"mean_absolute_error":float(absflat.mean()),"std_error":float(flat.std()),"median_absolute_error":float(np.median(absflat)),"min_error":float(flat.min()),"max_error":float(flat.max()),"percentiles":{str(q):float(np.percentile(absflat,q)) for q in [25,50,75,90,95,99]}}

np.save(RESULTS_ROOT/"predictions/predictions.npy",pred.astype(np.float32)); np.save(RESULTS_ROOT/"predictions/targets.npy",true.astype(np.float32)); np.save(RESULTS_ROOT/"predictions/errors.npy",err.astype(np.float32))
np.savez_compressed(RESULTS_ROOT/"predictions/predictions_targets_errors.npz",predictions=pred.astype(np.float32),targets=true.astype(np.float32),errors=err.astype(np.float32),sample_start_indices=starts)
pd.DataFrame([overall_ext]).to_csv(RESULTS_ROOT/"metrics/overall_metrics.csv",index=False); h_ext.to_csv(RESULTS_ROOT/"metrics/horizon_metrics.csv",index=False); n_ext.to_csv(RESULTS_ROOT/"metrics/node_metrics.csv",index=False); t_ext.to_csv(RESULTS_ROOT/"metrics/time_metrics.csv",index=False); pd.DataFrame([err_stats]).to_json(RESULTS_ROOT/"metrics/error_statistics.json",orient="records")
# Mechanically selected best/worst individual prediction points
flat_idx=np.argpartition(np.abs(err).reshape(-1),min(49,err.size-1))[:50]; worst_idx=np.argpartition(np.abs(err).reshape(-1),max(0,err.size-50))[-50:]
def cases(ids,reverse=False):
    rows=[]
    for q in ids:
        i,h,j=np.unravel_index(int(q),err.shape); y=float(true[i,h,j]); p=float(pred[i,h,j]); ae=abs(p-y)
        rows.append({"sample_index":i,"horizon_step":h+1,"sensor_id":sensor_ids[j],"timestamp":str(timestamps[starts[i]+h]),"actual":y,"predicted":p,"absolute_error":ae,"relative_error_percent":ae/max(abs(y),5.0)*100})
    return pd.DataFrame(rows).sort_values("absolute_error",ascending=reverse)
best_cases=cases(flat_idx); worst_cases=cases(worst_idx,True); best_cases.to_csv(RESULTS_ROOT/"metrics/best_cases.csv",index=False); worst_cases.to_csv(RESULTS_ROOT/"metrics/worst_cases.csv",index=False)

# Save key plots as files
def savefig(name):
    plt.tight_layout(); plt.savefig(RESULTS_ROOT/name,dpi=160,bbox_inches="tight"); plt.close()
plt.figure(figsize=(10,4.5)); plt.plot(h_ext["Horizon Minutes"],h_ext["MAE"],marker="o"); plt.xlabel("Minutes"); plt.ylabel("MAE"); plt.title("Horizon-wise MAE"); savefig("visualizations/horizons/horizon_mae.png")
plt.figure(figsize=(10,4.5)); plt.plot(h_ext["Horizon Minutes"],h_ext["RMSE"],marker="o"); plt.xlabel("Minutes"); plt.ylabel("RMSE"); plt.title("Horizon-wise RMSE"); savefig("visualizations/horizons/horizon_rmse.png")
plt.figure(figsize=(12,4)); plt.plot(n_ext["MAE"]); plt.xlabel("Sensor index"); plt.ylabel("MAE"); plt.title("Node-wise MAE"); savefig("visualizations/nodes/node_mae.png")
plt.figure(figsize=(12,4)); plt.plot(n_ext["RMSE"]); plt.xlabel("Sensor index"); plt.ylabel("RMSE"); plt.title("Node-wise RMSE"); savefig("visualizations/nodes/node_rmse.png")
plt.figure(figsize=(8,5)); plt.hist(flat,bins=60); plt.axvline(0,linestyle="--"); plt.xlabel("Residual"); plt.ylabel("Count"); plt.title("Residual distribution"); savefig("visualizations/distributions/residual_distribution.png")
plt.figure(figsize=(12,6)); plt.imshow(np.mean(np.abs(err),axis=0),aspect="auto"); plt.colorbar(label="MAE"); plt.xlabel("Sensor"); plt.ylabel("Horizon"); plt.title("Absolute-error heatmap"); savefig("visualizations/errors/absolute_error_heatmap.png")
plt.figure(figsize=(12,6)); plt.imshow(np.mean(err,axis=0),aspect="auto"); plt.colorbar(label="Residual"); plt.xlabel("Sensor"); plt.ylabel("Horizon"); plt.title("Residual heatmap"); savefig("visualizations/errors/residual_heatmap.png")

performance={"inference_seconds":float(elapsed),"batch_size":int(USED_BATCH_SIZE),"throughput_windows_per_second":float(len(X)/elapsed),"peak_gpu_memory_gb":None}
if torch.cuda.is_available(): performance["peak_gpu_memory_gb"]=float(torch.cuda.max_memory_allocated()/1024**3)
config_out={"dataset":DATASET,"checkpoint":str(MODEL_PATH),"seed":42,"device":str(device),"num_nodes":N,"input_steps":INPUT_STEPS,"horizon":HORIZON,"input_channels":INPUT_CHANNELS,"hidden":HIDDEN,"dropout":DROPOUT,"test_windows":len(X),"test_boundary_row":int(val_end),"normalization":"checkpoint train-period per-sensor z-score","mape_denominator_floor":5.0}
env={"python":sys.version,"pytorch":torch.__version__,"cuda_available":torch.cuda.is_available(),"cuda_version":torch.version.cuda,"gpu":torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}
report={"experiment":"CPSTN Draft 5 Testing","dataset":DATASET,"overall_metrics":overall_ext,"horizon_metrics":h_ext.to_dict(orient="records"),"error_statistics":err_stats,"performance":performance,"configuration":config_out,"environment":env}
for fn,obj in [("configuration.json",config_out),("environment_info.json",env),("testing_report.json",report)]: (RESULTS_ROOT/"reports"/fn).write_text(json.dumps(obj,indent=2,default=str),encoding="utf-8")
(RESULTS_ROOT/"reports/testing_report.txt").write_text("CPSTN DRAFT 5 TESTING REPORT\n"+json.dumps(report,indent=2,default=str),encoding="utf-8")
print("Comprehensive exports written to",RESULTS_ROOT)


In [ ]:
# Final package: CPSTN_Testing_Draft5_Results.zip
zip_path=Path("/content/CPSTN_Testing_Draft5_Results.zip")
if zip_path.exists(): zip_path.unlink()
readme=(RESULTS_ROOT/"README.txt")
readme.write_text("CPSTN Draft 5 Testing Results\n\nGenerated from actual execution of the supplied final checkpoint and compatible benchmark data. No fabricated metrics are included.\n\nMAPE follows the training convention with denominator floor 5.0. The checkpoint uses one observed speed feature, 12 input steps, 12 forecast steps, and 207 METR-LA sensors.\n",encoding="utf-8")
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for p in RESULTS_ROOT.rglob("*"):
        if p.is_file(): z.write(p,p.relative_to(RESULTS_ROOT.parent))
with zipfile.ZipFile(zip_path) as z:
    bad=z.testzip()
    if bad: raise RuntimeError(f"ZIP integrity failure: {bad}")
print("Results ZIP:",zip_path)
print("ZIP integrity: PASS")


## Appendix: Export Testing Artifacts

In [ ]:
# Export reproducible testing artifacts
pred_path = EXPORT_DIR / f"{DATASET}_CPSTN_Draft5_test_predictions.npz"
metrics_path = EXPORT_DIR / f"{DATASET}_CPSTN_Draft5_overall_metrics.csv"
horizon_path = EXPORT_DIR / f"{DATASET}_CPSTN_Draft5_horizon_metrics.csv"
sensor_path = EXPORT_DIR / f"{DATASET}_CPSTN_Draft5_sensor_metrics.csv"
route_path = EXPORT_DIR / f"{DATASET}_CPSTN_Draft5_route.csv"
metadata_path = EXPORT_DIR / f"{DATASET}_CPSTN_Draft5_metadata.json"

np.savez_compressed(
    pred_path,
    prediction=pred.astype(np.float32),
    target=true.astype(np.float32),
    prediction_z=pred_z.astype(np.float32),
    target_z=true_z.astype(np.float32),
    sample_start_indices=starts.astype(np.int64),
)

overall_df.to_csv(metrics_path, index=False)
horizon_df.to_csv(horizon_path, index=False)
sensor_df.to_csv(sensor_path, index=False)
route_df.to_csv(route_path, index=False)

metadata = {
    "dataset": DATASET,
    "model": str(ckpt.get("model_name", "CPSTN-Draft4")),
    "checkpoint": str(MODEL_PATH),
    "checkpoint_epoch": ckpt.get("epoch"),
    "best_validation_loss": ckpt.get("best_val_loss"),
    "num_nodes": N,
    "input_steps": INPUT_STEPS,
    "horizon": HORIZON,
    "input_channels": INPUT_CHANNELS,
    "hidden": HIDDEN,
    "dropout": DROPOUT,
    "test_split": "20% chronological",
    "train_fraction": 0.70,
    "validation_fraction": 0.10,
    "test_start_row": int(val_end),
    "test_first_target_row": int(first_t),
    "test_last_target_start_row": int(last_t),
    "num_test_windows": int(len(X)),
    "batch_size": int(USED_BATCH_SIZE),
    "inference_seconds": float(elapsed),
    "test_mae": overall["MAE"],
    "test_rmse": overall["RMSE"],
    "test_mape_percent": overall["MAPE_percent"],
    "route_source": sensor_ids[source],
    "route_destination": sensor_ids[destination],
    "route_horizon_step": route_h + 1,
    "route_sample_index": route_sample,
    "route_cost": float(route_cost),
    "route_estimated_travel_time": None,
    "route_cost_definition": "adjacency-weighted inverse predicted speed proxy",
    "a_star_heuristic": "zero; geographic coordinates unavailable",
    "sensor_exclusions": excluded_sensors,
}

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Artifacts will be written to:", EXPORT_DIR)
for p in sorted(EXPORT_DIR.glob("*")):
    print(" -", p.name)


# Authoritative final integrity check
assert MODEL_PATH.exists()
assert DATASET in EXPECTED_NODES and N == EXPECTED_NODES[DATASET]
assert X.shape[1:] == (INPUT_STEPS,N) and Y.shape[1:] == (HORIZON,N)
assert pred.shape == true.shape == (len(X),HORIZON,N)
assert np.isfinite(pred).all() and np.isfinite(true).all()
assert starts[0] == val_end and starts[-1] == T-HORIZON
assert params == 26668
print("FINAL INTEGRITY CHECK: PASS")


In [ ]:
print("="*68)
print("CPSTN TESTING COMPLETE")
print("="*68)
print("Model:", MODEL_PATH.name)
print("Dataset:", DATASET)
print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("Test samples:", len(X))
print("Prediction shape:", pred.shape)
print("Target shape:", true.shape)
print("Overall MAE:", f"{overall['MAE']:.6f}")
print("Overall RMSE:", f"{overall['RMSE']:.6f}")
print("Overall MAPE:", f"{overall['MAPE_percent']:.4f}%")
print("Inference time:", f"{elapsed:.4f} s")
print("Results directory:", RESULTS_ROOT)
print("Results ZIP:", "/content/CPSTN_Testing_Draft5_Results.zip")
print("Notebook:", "CPSTN(Testing)-Draft5.ipynb")
print("ALL TESTING RESULTS GENERATED SUCCESSFULLY")
print("="*68)
